In [1]:
import os
import shutil
import torch
import csv
from ultralytics import RTDETR, YOLO


def quantize_evaluate_and_log(model_name, weight_path, data_path, dataset_label, half=False):
    """
    Exports a model to ONNX, evaluates it on the test set, and returns the metrics.
    """
    if not os.path.exists(weight_path):
        print(f"\n[WARNING] PyTorch Weights not found at: {weight_path}")
        return None

    precision_str = "FP16" if half else "FP32"
    print(f"\n{'=' * 70}")
    print(f"EXPORTING & EVALUATING: {model_name} | Data: {dataset_label} | Format: ONNX {precision_str}")
    print(f"{'=' * 70}")

    # FIX 1: use replace('-', '') so 'RT-DETR-l' → 'rtdetrl' which contains 'rtdetr'
    is_rtdetr = 'rtdetr' in model_name.lower().replace('-', '')

    if is_rtdetr:
        model = RTDETR(weight_path)
        dynamic_export = True
    else:
        model = YOLO(weight_path)
        dynamic_export = False

    print(f"Exporting to ONNX ({precision_str})...")
    default_onnx_path = model.export(
        format='onnx',
        half=half,
        device=0,
        imgsz=640,
        simplify=True,
        dynamic=dynamic_export,
        opset=17
    )

    models_dir = os.path.dirname(weight_path)
    onnx_output_dir = os.path.join(models_dir, 'onnx_exports')
    os.makedirs(onnx_output_dir, exist_ok=True)

    base_name = os.path.basename(weight_path).replace('.pt', '')
    new_filename = f"{base_name}_{precision_str}.onnx"
    final_onnx_path = os.path.join(onnx_output_dir, new_filename)

    if os.path.exists(final_onnx_path):
        os.remove(final_onnx_path)
    shutil.move(default_onnx_path, final_onnx_path)

    print(f"Export moved and saved to: {final_onnx_path}")

    del model
    torch.cuda.empty_cache()

    print("Evaluating ONNX model on test set...")
    if is_rtdetr:
        onnx_model = RTDETR(final_onnx_path)
    else:
        onnx_model = YOLO(final_onnx_path)

    metrics = onnx_model.val(data=data_path, split='test', batch=16, workers=4, device=0, plots=False)

    # FIX 2: confirm which split was actually used
    actual_split = getattr(metrics, 'args', {})
    if hasattr(actual_split, 'get'):
        print(f"[INFO] Evaluated on split: {actual_split.get('split', 'unknown')}")

    # sanity check — warn if all metrics are zero
    if metrics.box.map50 == 0.0 and metrics.box.map == 0.0:
        print(f"[WARNING] All metrics are zero for {model_name} on {dataset_label} — check your dataset yaml and split.")

    results = {
        'Architecture': model_name,
        'Dataset': dataset_label,
        'Format': f'ONNX_{precision_str}',
        'Precision': round(metrics.box.mp, 4),
        'Recall': round(metrics.box.mr, 4),
        'mAP_50': round(metrics.box.map50, 4),
        'mAP_50_95': round(metrics.box.map, 4),
        'Inference_Time_ms': round(metrics.speed['inference'], 2),
        'FPS': round(1000 / metrics.speed['inference'], 2) if metrics.speed['inference'] > 0 else 0
    }

    del onnx_model
    torch.cuda.empty_cache()

    return results


def main():
    BASE_DIR = r'C:\Users\USER\Desktop\py'
    HITUAV_DATA = os.path.join(BASE_DIR, r'../Data/hit-uav/dataset.yaml')
    VISDRONE_DATA = os.path.join(BASE_DIR, r'data\VisDrone\VisDrone.yaml')
    MODELS_DIR = os.path.join(BASE_DIR, '../models')

    best_models = [
        ('YOLO26s',   os.path.join(MODELS_DIR, r'yolo_IR.pt'),   HITUAV_DATA,   'IR'),
        ('RT-DETR-l', os.path.join(MODELS_DIR, r'rtdtr_IR.pt'),  HITUAV_DATA,   'IR'),
        ('YOLO26s',   os.path.join(MODELS_DIR, r'yolo_RGB.pt'),  VISDRONE_DATA, 'RGB'),
        ('RT-DETR-l', os.path.join(MODELS_DIR, r'rtdtr_RGB.pt'), VISDRONE_DATA, 'RGB'),
    ]

    all_results = []

    for is_half_precision in [True, False]:
        for name, weight_path, data_path, label in best_models:
            try:
                res = quantize_evaluate_and_log(
                    model_name=name,
                    weight_path=weight_path,
                    data_path=data_path,
                    dataset_label=label,
                    half=is_half_precision
                )
                if res:
                    all_results.append(res)
            except Exception as e:
                print(f"[ERROR] Failed for {name} | {label} | half={is_half_precision}: {e}")

    csv_file = os.path.join(BASE_DIR, '../onnx_evaluation_metrics.csv')
    if all_results:
        keys = all_results[0].keys()
        with open(csv_file, 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, fieldnames=keys)
            dict_writer.writeheader()
            dict_writer.writerows(all_results)
        print(f"\nAll ONNX evaluations complete! Metrics saved to: {csv_file}")
    else:
        print("\nNo models were successfully evaluated.")


if __name__ == '__main__':
    main()


EXPORTING & EVALUATING: YOLO26s | Data: IR | Format: ONNX FP16
Exporting to ONNX (FP16)...
Ultralytics 8.4.49  Python-3.10.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from 'C:\Users\USER\Desktop\py\models\yolo_IR.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


c:\Users\USER\Desktop\py\myenv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset9.py:5353: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success  3.9s, saved as 'C:\Users\USER\Desktop\py\models\yolo_IR.onnx' (18.3 MB)

Export complete (4.9s)
Results saved to C:\Users\USER\Desktop\py\models\yolo_IR.onnx
Predict:         yolo predict task=detect model=C:\Users\USER\Desktop\py\models\yolo_IR.onnx imgsz=640 half
Validate:        yolo val task=detect model=C:\Users\USER\Desktop\py\models\yolo_IR.onnx imgsz=640 data=C:\Users\User\PythonProject\cv_proj\Data\hit-uav\dataset.yaml half 
Visualize:       https://netron.app
Export moved and saved to: C:\Users\USER\Desktop\py\models\onnx_exports\yolo_IR_FP16.onnx
Evaluating ONNX model on test set...
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.49  Python-3.10.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Loading C:\Users\USER\Desktop\py\models\onnx_exports\yolo_IR_

c:\Users\USER\Desktop\py\myenv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\jit_utils.py:303: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
c:\Users\USER\Desktop\py\myenv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset9.py:5353: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(
c:\Users\USER\Desktop\py\myenv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:711: UserWarning: Constant folding - Only steps=1 can b

ONNX: slimming with onnxslim 0.1.93...


c:\Users\USER\Desktop\py\myenv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:1181: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_graph_shape_type_inference(


ONNX: export success  9.8s, saved as 'C:\Users\USER\Desktop\py\models\rtdtr_IR.onnx' (62.8 MB)

Export complete (11.1s)
Results saved to C:\Users\USER\Desktop\py\models\rtdtr_IR.onnx
Predict:         yolo predict task=detect model=C:\Users\USER\Desktop\py\models\rtdtr_IR.onnx imgsz=640 half
Validate:        yolo val task=detect model=C:\Users\USER\Desktop\py\models\rtdtr_IR.onnx imgsz=640 data=C:\Users\User\PythonProject\cv_proj\Data\hit-uav\dataset.yaml half 
Visualize:       https://netron.app
Export moved and saved to: C:\Users\USER\Desktop\py\models\onnx_exports\rtdtr_IR_FP16.onnx
Evaluating ONNX model on test set...
Ultralytics 8.4.49  Python-3.10.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Loading C:\Users\USER\Desktop\py\models\onnx_exports\rtdtr_IR_FP16.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CUDAExecutionProvider
val: Fast image access  (ping: 0.10.0 ms, read: 512.2114.4 MB/s, size: 66.9 KB)
val: Scanning C:\Users\USER\Des